# CS5830 Project 7 - Support Vector Machines and Logistic Regression

## Emberly's Work

In [ ]:
import pandas as pd
import numpy as np
import random
import seaborn as sns

import matplotlib.pyplot as plt

df = pd.read_csv('data/austin_animal_outcomes.csv')

# calculate the total number of animals, and the animals that have been adopted
total_animals = len(df)
adopted_animals = (df['Outcome Status'] == "Adopted").sum()
percentage = (adopted_animals / total_animals) * 100

print(f"Total Animals: {total_animals}")
print(f"Adopted Animals: {adopted_animals}")
print(f"Percentage Adopted: {percentage:.2f}%")

# visualize the distribution of animal outcomes with readable labels
plt.figure(figsize=(10, 6))
sns.countplot(x='Outcome Status', data=df)
plt.title('Distribution of Animal Outcomes')
plt.xlabel('Outcome Status')
plt.ylabel('Count')

# Rotate labels 45 degrees and align them to the right of their tick mark
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()



In [ ]:
# Drop Reclaimed animals, since they didn't need to be adopted
df = df[df['Outcome Status'] != "Reclaimed"]
print(f"Total Animals after removing Reclaimed animals: {len(df)}") 

# Drop animals that don't have an outcome
df = df[df['Outcome Status'].notna()]
print(f"Total Animals after removing animals without an outcome: {len(df)}")

# Drop duplicates
df = df.drop_duplicates()
print(f"Total Animals after removing duplicates: {len(df)}")

# split up adopted vs not adopted animals
adopted_df = df[(df['Outcome Status'].str.contains("Adopted"))]
not_adopted_df = df[~df['Outcome Status'].str.contains("Adopted")]
print(f"Adopted Animals: {len(adopted_df)}")
print(f"Not Adopted Animals: {len(not_adopted_df)}")
print(f"Percentage Adopted: {(len(adopted_df) / len(df)) * 100:.2f}%")
print(f"Percentage Not Adopted: {(len(not_adopted_df) / len(df)) * 100:.2f}%")

# Create a new column indicating whether the animal was adopted
df['Adopted'] = df['Outcome Status'].str.contains("Adopted")

# visualize the distribution of adopted vs not adopted animals
plt.figure(figsize=(10, 6))
sns.countplot(x='Adopted', data=df)
plt.title('Distribution of Adopted vs Not Adopted Animals')
plt.xlabel('Adopted')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# List of factors that I think might influence whether an animal is adopted (all currently strings)

# Days in Shelter (quant)
# Date of Birth in relation to Outcome Date (show how old they were when they had an outcome) (quant)
# Type (qual)
# Primary Breed (qual)
# has a date of birth (bool)

# Clean and prepare the date of birth and age at outcome data for analysis
df['Date of Birth'] = pd.to_datetime(df['Date of Birth'], errors='coerce')
df['Outcome Date'] = pd.to_datetime(df['Outcome Date'], errors='coerce')
df['has_date_of_birth'] = df['Date of Birth'].notna().astype(int)
df['Age at Outcome'] = (df['Outcome Date'] - df['Date of Birth']).dt.days / 365.25
# 4. Fill missing values with a placeholder (e.g., -1)
df['Age at Outcome'] = df['Age at Outcome'].fillna(-1)
# Add a boolean column for "has a Date of Birth"
df['Has Date of Birth'] = df['Date of Birth'].notna()


# cleaning the columns for analysis and dropping NaN values
df['Days in Shelter'] = pd.to_numeric(df['Days in Shelter'], errors='coerce')
df['Type'] = df['Type'].astype(str)
df['Primary Breed'] = df['Primary Breed'].astype(str)
df = df.dropna(subset=['Days in Shelter', 'Type', 'Primary Breed', 'Outcome Date'])
print(f"Total Animals after dropping rows with NaN values in key columns: {len(df)}")

# visualize the distribution of adopted vs not adopted animals
plt.figure(figsize=(10, 6))
sns.countplot(x='Adopted', data=df)
plt.title('Distribution of Adopted vs Not Adopted Animals')
plt.xlabel('Adopted')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Preparing for SVM Analysis by one hot encoding the qualitative features based on high vs low cardinality
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Handle 'Primary Breed' which is a high cardinality feature by one hot encoding it
threshold = 20
primary_breed_counts = df['Primary Breed'].value_counts()
rare_breeds = primary_breed_counts[primary_breed_counts < threshold].index
df['Primary Breed Cleaned'] = df['Primary Breed'].replace(rare_breeds, 'Other')

# Define feature groups
categorical_cols = ['Type', 'Primary Breed Cleaned'] # Low & medium cardinality
numerical_cols = ['Days in Shelter', 'Age at Outcome', 'has_date_of_birth'] # Includes binary flag

# Build Preprocessing ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols) # Scales all numerical columns (including 1/0 flag)
    ]
)

# Prepare X (features) and y (target: 'Adopted')
X = df[['Type', 'Primary Breed Cleaned', 'Days in Shelter', 'Age at Outcome', 'has_date_of_birth']]
y = df['Adopted']

# Transform data ready for SVM
X_processed = preprocessor.fit_transform(X)

# Train/Test Split
# Stratify ensures the proportion of adopted/not adopted animals is the same in both splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Create a full SVM pipeline including preprocessing and classification
# class_weight='balanced' is vital for shelter data to account for class imbalances
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(
        kernel='rbf', 
        C=1.0,           
        gamma=0.8,       
        class_weight='balanced', 
        random_state=42
    ))
])

# Train and Evaluate the SVM pipeline
svm_pipeline.fit(X_train, y_train)

y_pred = svm_pipeline.predict(X_test)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Create a Logistic Regression pipeline including preprocessing and classification
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        C=1.0,
        class_weight='balanced',
        random_state=42
    ))
])

# Train and Evaluate the Logistic Regression pipeline
logreg_pipeline.fit(X_train, y_train)

y_pred_logreg = logreg_pipeline.predict(X_test)

print("Accuracy Score:", accuracy_score(y_test, y_pred_logreg))
print("\nClassification Report:\n", classification_report(y_test, y_pred_logreg))

## Sulove's Work

In [ ]:
# put work here